# 535. Encode and Decode TinyURL

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** design, hash-table, string
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/encode-and-decode-tinyurl/)

TinyURL is a URL shortening service where you enter a URL such as
`https://leetcode.com/problems/design-tinyurl` and it returns a short one such as
`http://tinyurl.com/4e9iAk`. Design a class to encode a URL and decode a tiny URL.

There is no restriction on how your encode/decode algorithm should work. You just
need to ensure that a URL can be encoded to a tiny URL and the tiny URL can be
decoded to the original URL.

Implement the `Codec` class:

- `encode(longUrl)` returns a tiny URL for the given `longUrl`.
- `decode(shortUrl)` returns the original long URL for the given `shortUrl`. It is
  guaranteed that the given `shortUrl` was encoded by the same object.

---

### Example

```
Input:  url = "https://leetcode.com/problems/design-tinyurl"
Output: "https://leetcode.com/problems/design-tinyurl"

Codec codec = new Codec();
codec.decode(codec.encode(url));   // returns the original url
```

---

### Constraints

- `1 <= url.length <= 10^4`
- `url` is guaranteed to be a valid URL

This is the first problem in the repo with **no correct answer** - only a spectrum
of answers with different properties. Read the requirement again: it says
`decode(encode(url)) == url` and *nothing else*. Not that it is short. Deciding what
the problem actually is, before solving it, is the whole exercise.

## Before you write anything

**1.** **The laziest legal answer.** `encode` returns the URL unchanged and `decode`
returns its argument. It satisfies every word of the specification and passes every
test LeetCode has. Write it in two lines, then write one sentence explaining why you
would not say it in an interview - and make the sentence about *the requirement that
was left out*, not about cheating.

**2.** So the real requirement is "short". Three ways to make a key:

```
(a) a counter:     1, 2, 3, ...        (b) random characters       (c) a hash of the URL
```

For each, answer: can two different URLs ever get the same key? What does that cost
the user when it happens? Rank the three by *how bad the failure is*, not by how
likely it is - and notice that (c)'s failure is a user silently landing on somebody
else's website.

**3.** Do you need one dict or two? `code -> url` is obviously required by `decode`.
What does the second one, `url -> code`, buy you - and what does
`encode(same_url_twice)` return with it and without it? Neither answer is wrong.
Say which one a real shortener wants, and why the other one is what an *analytics*
product wants.

**4.** Your counter produces `1, 2, 3`. Real short codes look like `4e9iAk`. That is
the same number written in **base 62** - `0-9`, `a-z`, `A-Z`. How many URLs fit in a
6-character base-62 code? Compute it, then compare with the number of URLs in the
world, and decide whether 6 characters was a good choice.

**5.** How do you test something whose output you deliberately did not specify? You
cannot compare against an expected string - any string is legal. So you test the
**property** instead: for every URL, `decode(encode(url))` must equal `url`. Name the
two other properties worth checking. (One is about two different URLs; one is about
the same URL twice.)

## Two routes

**A - counter plus one dict** *(write this first)*

```
self.urls = {}       # code -> long url
self.next_id = 1
```

`encode` stores the URL under `str(self.next_id)`, increments, and returns
`"http://tinyurl.com/" + code`. `decode` splits the code off the end and looks it up.

Collisions are **impossible**, not unlikely - the counter never repeats. That is the
property that makes this the right shape, and it is why real shorteners are built on
a counter rather than on a hash. `O(1)` for both, memory `O(n)` in URLs stored.

The one thing to get right is `decode`: take the part after the **last** `/`, not
`split("/")[-1]` on an assumption about how many slashes there are. Write it so a
prefix change does not break it.

**B - base 62, and a second dict for deduplication**

Convert the counter to base 62 so `12345` becomes `dnh` instead of `"12345"` - the
same information, 60% fewer characters, and it looks like a real short link. Then
add `self.codes = {}` mapping `url -> code`, so encoding the same URL twice hands
back the same short link instead of burning a new ID.

That second dict is a genuine product decision, not an optimisation. With it, one
URL has one short link forever. Without it, every request mints a new link - which is
exactly what you want if each link is a *campaign* you need to count clicks on
separately. Say which one you built, and why.

> **The specification is the problem.** Every other exercise in this repo has one
> right answer; this one has a requirement that was deliberately left off the page,
> and your first job is to notice the gap and write it down. In real work that is
> most of the job.

In [ ]:
class Codec:

    def __init__(self):
        pass

    def encode(self, longUrl: str) -> str:
        '''Encodes a URL to a shortened URL.'''
        pass

    def decode(self, shortUrl: str) -> str:
        '''Decodes a shortened URL to its original URL.'''
        pass

### The test harness

Question 5's answer. The output is not specified, so there is nothing to compare
against - the harness checks **properties** instead:

1. **Round trip.** `decode(encode(url)) == url`, for every URL, including nasty ones.
2. **Distinctness.** Two different URLs must not encode to the same short URL. That
   is the collision from question 2, and it is the failure that silently sends a user
   to the wrong site.
3. **Interleaving.** Encode a hundred URLs *first*, then decode them all in a
   different order. A `Codec` that keeps only the most recent URL in a single
   variable passes a naive round-trip test and dies here.
4. **Shortness**, reported but never failed. The harness prints the average short-URL
   length next to the average input length. Question 1's two-line cheat scores 1.00x
   and passes everything - which is the point, and why the number is printed where
   you have to look at it.

`stress` generates long random URLs with query strings and fragments. Run this cell;
don't edit it.

In [ ]:
import random
import string


def check(urls, shuffle_seed=None):
    '''Property-test a Codec over a batch of urls: round trip, distinctness, interleaving.'''
    log = []
    try:
        codec = Codec()
    except Exception as e:
        return False, [f"   !! Codec() raised {type(e).__name__}: {e}"], None

    # encode everything FIRST - a Codec with one slot of memory dies here
    codes = []
    for u in urls:
        try:
            c = codec.encode(u)
        except Exception as e:
            log.append(f"   !! encode({u[:40]!r}...) raised {type(e).__name__}: {e}")
            return False, log, None
        if not isinstance(c, str):
            log.append(f"   !! encode must return a str, got {type(c).__name__}: {c!r}")
            return False, log, None
        codes.append(c)

    # distinctness: different urls must not share a code
    seen = {}
    for u, c in zip(urls, codes):
        if c in seen and seen[c] != u:
            log.append(f"   !! COLLISION - two different urls encoded to {c!r}:")
            log.append(f"      {seen[c][:60]}")
            log.append(f"      {u[:60]}")
            return False, log, None
        seen[c] = u

    # decode in a different order than we encoded
    order = list(range(len(urls)))
    if shuffle_seed is not None:
        random.Random(shuffle_seed).shuffle(order)

    for i in order:
        try:
            back = codec.decode(codes[i])
        except Exception as e:
            log.append(f"   !! decode({codes[i]!r}) raised {type(e).__name__}: {e}")
            return False, log, None
        if back != urls[i]:
            log.append(f"   !! round trip failed for {codes[i]!r}")
            log.append(f"      encoded  {urls[i][:60]}")
            log.append(f"      decoded  {str(back)[:60]}")
            return False, log, None

    ratio = sum(len(c) for c in codes) / max(1, sum(len(u) for u in urls))
    stats = (len(urls), sum(len(u) for u in urls) / len(urls),
             sum(len(c) for c in codes) / len(codes), ratio)
    log.append(f"{len(urls)} urls round-tripped, all codes distinct")
    return True, log, stats


def report(name, result):
    ok, log, stats = result
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if ok and stats:
        n, avg_in, avg_out, ratio = stats
        print(f"       {n:>4} urls   avg input {avg_in:6.1f} chars"
              f"   avg short {avg_out:6.1f} chars   ratio {ratio:5.2f}x")
    if not ok:
        for line in log[-5:]:
            print(f"       {line}")


def random_urls(n, seed=0):
    random.seed(seed)
    out = []
    for _ in range(n):
        path = "/".join("".join(random.choices(string.ascii_lowercase, k=random.randint(3, 12)))
                        for _ in range(random.randint(1, 4)))
        q = "".join(random.choices(string.ascii_lowercase + string.digits + "=&",
                                   k=random.randint(0, 30)))
        out.append(f"https://example.com/{path}" + (f"?{q}" if q else ""))
    return out

In [ ]:
# tests
report("the LeetCode example",
       check(["https://leetcode.com/problems/design-tinyurl"]))

report("two different urls (distinctness)",
       check(["https://a.com/x", "https://a.com/y"]))

report("the same url encoded twice - both codes must still decode",
       check(["https://a.com/x", "https://a.com/x"]))

report("urls that differ by one character",
       check([f"https://a.com/page{i}" for i in range(50)]))

report("urls containing slashes, queries and fragments",
       check(["https://a.com/a/b/c/d/e",
              "https://a.com/search?q=hello+world&page=2",
              "https://a.com/docs#section-4",
              "https://a.com/a/b/c/d/e?x=1#top"]))

report("a url that looks like one of your own short urls",
       check(["http://tinyurl.com/1", "http://tinyurl.com/2", "https://real.com/page"]))

report("the maximum length url (10^4 chars)",
       check(["https://a.com/" + "x" * 9986]))

report("100 urls, decoded in a shuffled order",
       check(random_urls(100, seed=1), shuffle_seed=7))

report("1000 urls, decoded in a shuffled order",
       check(random_urls(1000, seed=2), shuffle_seed=8))

report("2000 urls - watch the ratio column",
       check(random_urls(2000, seed=3), shuffle_seed=9))

## After it passes

- **Look at the ratio column.** Route A stores `"http://tinyurl.com/12345"` - that is
  24 characters of which 19 are the same every time. Build route B's base-62 codec and
  compare the two ratios on the 2000-url case. Then ask the question that matters: at
  what input length does your "short" URL stop being shorter than the original?
- **Compute the space.** `62^6` codes - write the number down. Then `62^7`. Then look
  up how many characters a real `bit.ly` code has and work out what they decided.
- **Deliberately collide.** Write a third codec that keys on `hash(url) % 10000` and run
  the 2000-url case. Watch the harness catch it, and note *which* test caught it -
  distinctness, not round-trip. A round-trip-only test suite would have shipped it.
- **The invariant**, and it is one line: *every code in `self.urls` maps to the URL that
  produced it, and no code maps to two URLs.* Both of the harness's first two checks
  exist to defend exactly that sentence.
- **Make it real.** Three features, in order of how much they change the design:
  expiry (a code stops working after 30 days - now you need #359's timestamp per
  entry), click counting (an integer per code, and suddenly the `url -> code` dedupe
  dict from route B becomes *wrong*), and multiple servers (two machines both handing
  out ID 1000 - what has to change about the counter?). That third one is why real
  shorteners do not use a plain counter.
- Siblings: #271 Encode and Decode Strings (the same "you design the format" freedom,
  but with an actual parsing trap), #146 LRU Cache (what to do when the dict must not
  grow forever), #1396 Design Underground System (two dicts that mean different
  things, like route B's pair).